# 🚀 Scraper de Leads Commerciaux - Version Avancée

Ce notebook scrape des leads depuis **Pages Jaunes**, **Bing Local** et **OpenCorporates**, les nettoie, les score et les exporte en CSV.

## Problèmes résolus:
- ✅ Gestion des DataFrames vides (KeyError 'score')
- ✅ Sources multiples de données
- ✅ Meilleure gestion d'erreur
- ✅ Logging détaillé pour déboguer
- ✅ Validation robuste des données

In [ ]:
# 1. IMPORTATION ET CONFIGURATION

#!pip install pandas requests beautifulsoup4 tqdm lxml

import requests
import pandas as pd
import os
import time
import re
import logging
from bs4 import BeautifulSoup
from urllib.parse import urljoin, quote
from datetime import datetime
from tqdm import tqdm
from typing import List, Dict

# Configuration du logging
logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')
logger = logging.getLogger(__name__)

# Try to mount Google Drive if in Colab
try:
    from google.colab import drive
    drive.mount('/content/drive/', force_remount=False)
    csv_dir = '/content/drive/My Drive/Colab Notebooks/leads_webexa'
except:
    csv_dir = './leads_export'

os.makedirs(csv_dir, exist_ok=True)

csv_path = os.path.join(
    csv_dir,
    f"leads_scrapped_{datetime.now().strftime('%Y%m%d_%H%M%S')}.csv"
)

logger.info(f"📁 Dossier de sauvegarde: {csv_dir}")

In [ ]:
# 2. CONFIGURATION DES SECTEURS ET VILLES

SECTEURS_CIBLES = [
    "agence immobilière", "expert-comptable", "transport routier", "logistique",
    "cabinet conseil", "hôtel restaurant", "cabinet médical", "clinique",
    "entreprise de nettoyage", "agence de voyage", "auto-école", "architecte",
    "plomberie", "électricité", "peinture", "menuiserie", "assurance", "banque",
    "entreprise de construction"
]

VILLES_CIBLES = [
    "Paris", "Marseille", "Lyon", "Toulouse", "Nice",
    "Nantes", "Strasbourg", "Montpellier", "Bordeaux", "Lille"
]

DELAI_REQUETES = 2  # secondes entre les requêtes

# Headers HTTP pour éviter les blocages
HEADERS = {
    'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36',
    'Accept': 'text/html,application/xhtml+xml,application/xml;q=0.9,*/*;q=0.8',
    'Accept-Language': 'fr-FR,fr;q=0.9,en;q=0.8',
}

print(f"✓ {len(SECTEURS_CIBLES)} secteurs configurés")
print(f"✓ {len(VILLES_CIBLES)} villes configurées")
print(f"✓ Délai entre requêtes: {DELAI_REQUETES}s")

In [ ]:
# 3. FONCTION DE CALCUL DE SCORE COMMERCIAL

def calcul_score(lead: Dict) -> int:
    """Calcule un score commercial de 0-100"""
    score = 0
    
    # Secteur ciblé (+20)
    secteur = (lead.get("secteur", "") or "").lower()
    mots_cles = ["immobilier", "comptab", "transport", "logistique", "conseil", 
                 "restaurant", "médical", "clinique", "nettoyage", "voyage"]
    if any(mot in secteur for mot in mots_cles):
        score += 20
    
    # Téléphone présent (+15)
    if lead.get("telephone"):
        score += 15
    
    # Email présent (+15)
    if lead.get("email"):
        score += 15
    
    # Site web présent (+10)
    if lead.get("site"):
        score += 10
    
    # Adresse complète (+10)
    if lead.get("adresse"):
        score += 10
    
    # Code postal (+5)
    if lead.get("code_postal"):
        score += 5
    
    return min(score, 100)

# Test
test_lead = {"secteur": "expert-comptable", "telephone": "0123456789", "email": "test@test.com"}
print(f"✓ Test score: {calcul_score(test_lead)}/100")

In [ ]:
# 4. SCRAPER PAGES JAUNES AMÉLIORÉ

def scraper_pages_jaunes(secteur: str, ville: str) -> List[Dict]:
    """Scrape Pages Jaunes avec meilleure gestion d'erreur"""
    resultats = []
    url_base = "https://www.pagesjaunes.fr/search"
    
    try:
        for page in range(1, 3):  # 2 pages max
            params = {
                "quoi": secteur,
                "ou": ville,
                "page": page
            }
            
            try:
                response = requests.get(
                    url_base,
                    params=params,
                    headers=HEADERS,
                    timeout=15
                )
                
                if response.status_code != 200:
                    logger.warning(f"Pages Jaunes: Status {response.status_code} pour {secteur}/{ville}")
                    continue
                
                soup = BeautifulSoup(response.content, 'html.parser')
                
                # Amélioration: chercher par plusieurs sélecteurs possibles
                items = soup.find_all(['li', 'div'], class_=re.compile(r'(result|company|business)', re.I))
                
                if not items:
                    logger.debug(f"Pages Jaunes: Aucun item trouvé pour {secteur}/{ville}")
                
                for item in items[:10]:  # Max 10 par page
                    try:
                        # Nom - chercher plusieurs balises possibles
                        nom_elem = item.find(['h2', 'h3', 'a'], class_=re.compile(r'(name|title)', re.I))
                        nom = nom_elem.get_text(strip=True) if nom_elem else ""
                        
                        if not nom or len(nom) < 3:
                            continue
                        
                        # Adresse
                        adresse_elem = item.find('div', class_=re.compile(r'address', re.I))
                        adresse = adresse_elem.get_text(strip=True) if adresse_elem else ""
                        
                        # Téléphone
                        tel_text = item.get_text()
                        tel_match = re.search(r'0[1-9](?:\s?\d{2}){4}', tel_text)
                        telephone = tel_match.group(0).replace(' ', '') if tel_match else ""
                        
                        # Email
                        email_match = re.search(r'[\w\.-]+@[\w\.-]+\.\w+', item.get_text())
                        email = email_match.group(0) if email_match else ""
                        
                        # Site web
                        site_elem = item.find('a', href=re.compile(r'https?://'))
                        site = site_elem.get('href', '') if site_elem else ""
                        
                        # Code postal
                        code_postal = re.search(r'\b(\d{5})\b', adresse).group(1) if adresse and re.search(r'\b(\d{5})\b', adresse) else ""
                        
                        if nom:
                            lead = {
                                "source": "Pages Jaunes",
                                "nom": nom[:100],  # Limiter la taille
                                "secteur": secteur,
                                "adresse": adresse[:200],
                                "code_postal": code_postal,
                                "ville": ville,
                                "telephone": telephone,
                                "email": email,
                                "site": site,
                                "date_scrape": datetime.now().isoformat()
                            }
                            lead["score"] = calcul_score(lead)
                            resultats.append(lead)
                    
                    except Exception as e:
                        logger.debug(f"Erreur parsing item: {e}")
                        continue
                
                time.sleep(DELAI_REQUETES)
            
            except requests.exceptions.RequestException as e:
                logger.warning(f"Pages Jaunes - Erreur requête: {e}")
                continue
    
    except Exception as e:
        logger.error(f"Pages Jaunes {secteur}/{ville}: {e}")
    
    return resultats

# Test
logger.info("Test de scraping Pages Jaunes...")

In [ ]:
# 5. SCRAPER API GOUVERNEMENTALE FRANÇAISE (MEILLEURE SOURCE)

def scraper_api_gouv(secteur: str, ville: str) -> List[Dict]:
    """Scrape l'API officielle gouvernementale française - TRÈS FIABLE"""
    resultats = []
    
    try:
        # L'API Gouv accepte: nom entreprise, code postal, etc.
        # Format: https://recherche-entreprises.api.gouv.fr/search?q=...
        
        # Chercher d'abord par secteur ET ville
        query = f"{secteur} {ville}"
        
        params = {
            "q": query,
            "per_page": 50,
            "page": 1
        }
        
        logger.debug(f"API Gouv request: {query}")
        
        response = requests.get(
            "https://recherche-entreprises.api.gouv.fr/search",
            params=params,
            timeout=15,
            headers=HEADERS
        )
        
        logger.debug(f"API Gouv URL: {response.url}")
        logger.debug(f"API Gouv Status: {response.status_code}")
        
        if response.status_code == 200:
            data = response.json()
            results = data.get("results", [])
            
            logger.info(f"✅ API Gouv: {len(results)} résultats pour {secteur}/{ville}")
            
            for company in results:
                try:
                    nom = company.get("nom_complet", "") or company.get("nom_entreprise", "")
                    if not nom or len(nom) < 3:
                        continue
                    
                    # Extraire l'adresse
                    adresse_parts = []
                    if company.get("adresse"):
                        adresse_parts.append(company.get("adresse", ""))
                    if company.get("code_postal"):
                        adresse_parts.append(company.get("code_postal", ""))
                    if company.get("commune"):
                        adresse_parts.append(company.get("commune", ""))
                    
                    adresse = ", ".join(filter(None, adresse_parts))
                    
                    lead = {
                        "source": "API Gouvernementale 🇫🇷",
                        "nom": nom[:100],
                        "secteur": secteur,
                        "adresse": adresse[:200],
                        "code_postal": company.get("code_postal", ""),
                        "ville": company.get("commune", ville),
                        "telephone": company.get("telephone", ""),
                        "email": company.get("email", ""),
                        "site": company.get("site_web", ""),
                        "date_scrape": datetime.now().isoformat()
                    }
                    
                    lead["score"] = calcul_score(lead)
                    resultats.append(lead)
                
                except Exception as e:
                    logger.debug(f"Erreur parsing API Gouv: {e}")
                    continue
        elif response.status_code == 400:
            logger.debug(f"⚠️  API Gouv 400 - Query invalide: '{query}'")
            # Essayer avec juste le secteur
            try:
                params2 = {
                    "q": secteur,
                    "per_page": 30
                }
                response2 = requests.get(
                    "https://recherche-entreprises.api.gouv.fr/search",
                    params=params2,
                    timeout=15,
                    headers=HEADERS
                )
                if response2.status_code == 200:
                    data = response2.json()
                    results = data.get("results", [])
                    logger.info(f"✅ API Gouv (fallback): {len(results)} pour {secteur}")
                    
                    # Filtrer par ville
                    for company in results:
                        if company.get("commune", "").lower() == ville.lower():
                            nom = company.get("nom_complet", "") or company.get("nom_entreprise", "")
                            if nom and len(nom) >= 3:
                                lead = {
                                    "source": "API Gouvernementale 🇫🇷",
                                    "nom": nom[:100],
                                    "secteur": secteur,
                                    "adresse": company.get("adresse", "")[:200],
                                    "code_postal": company.get("code_postal", ""),
                                    "ville": company.get("commune", ville),
                                    "telephone": company.get("telephone", ""),
                                    "email": company.get("email", ""),
                                    "site": company.get("site_web", ""),
                                    "date_scrape": datetime.now().isoformat()
                                }
                                lead["score"] = calcul_score(lead)
                                resultats.append(lead)
            except Exception as e:
                logger.debug(f"Fallback API Gouv échoué: {e}")
        else:
            logger.warning(f"⚠️  API Gouv {response.status_code}: {secteur}/{ville}")
        
        time.sleep(0.5)  # Rate limit doux pour API officielle
    
    except Exception as e:
        logger.error(f"API Gouv {secteur}/{ville}: {e}")
    
    return resultats


def scraper_opencorporates_simple(secteur: str, ville: str) -> List[Dict]:
    """Scraper OpenCorporates SANS clé API (limite: résultats limités)"""
    resultats = []
    
    try:
        # OpenCorporates API publique (très limité sans clé)
        query = f"{secteur} {ville} France"
        
        params = {
            "q": query,
            "jurisdiction_code": "fr",
            "per_page": 10
        }
        
        response = requests.get(
            "https://api.opencorporates.com/v0.4/companies/search",
            params=params,
            timeout=15,
            headers=HEADERS
        )
        
        if response.status_code == 200:
            data = response.json()
            companies = data.get("results", {}).get("companies", [])
            
            if companies:
                logger.info(f"✅ OpenCorporates: {len(companies)} résultats pour {secteur}/{ville}")
            
            for company in companies:
                try:
                    comp = company.get("company", {})
                    nom = comp.get("name", "")
                    
                    if not nom or len(nom) < 3:
                        continue
                    
                    lead = {
                        "source": "OpenCorporates",
                        "nom": nom[:100],
                        "secteur": secteur,
                        "adresse": comp.get("registered_address_in_full", "")[:200],
                        "code_postal": "",
                        "ville": ville,
                        "telephone": "",
                        "email": "",
                        "site": "",
                        "date_scrape": datetime.now().isoformat()
                    }
                    
                    # Extraire code postal si dans adresse
                    adresse = lead["adresse"]
                    if adresse:
                        cp_match = re.search(r'\b(\d{5})\b', adresse)
                        if cp_match:
                            lead["code_postal"] = cp_match.group(1)
                    
                    lead["score"] = calcul_score(lead)
                    resultats.append(lead)
                
                except Exception as e:
                    logger.debug(f"Erreur parsing OpenCorporates: {e}")
                    continue
        elif response.status_code == 401:
            logger.debug(f"⚠️  OpenCorporates 401 - API key requise (skipped)")
        else:
            logger.debug(f"OpenCorporates {response.status_code}")
        
        time.sleep(0.5)
    
    except Exception as e:
        logger.debug(f"OpenCorporates {secteur}/{ville}: {e}")
    
    return resultats

logger.info("✓ Scrapers configurés: API Gouv (primaire) + OpenCorporates (fallback léger)")

In [ ]:
# 6. NETTOYAGE ET VALIDATION DES DONNÉES

def nettoyer_lead(lead: Dict) -> Dict:
    """Nettoie et valide les données du lead"""
    
    # Supprimer espaces inutiles
    for key in ['nom', 'adresse', 'email', 'telephone', 'site', 'secteur', 'ville']:
        if key in lead:
            lead[key] = (lead[key] or "").strip()
    
    # Valider email (regex simple)
    if lead.get('email'):
        if not re.match(r'^[\w\.-]+@[\w\.-]+\.\w+$', lead['email']):
            lead['email'] = ""
    
    # Valider téléphone (10 chiffres pour France)
    if lead.get('telephone'):
        digits = re.sub(r'\D', '', lead['telephone'])
        if len(digits) >= 10:
            lead['telephone'] = digits[-10:]
        else:
            lead['telephone'] = ""
    
    # Normaliser URL
    if lead.get('site'):
        if not lead['site'].startswith('http'):
            lead['site'] = 'https://' + lead['site']
    
    return lead

def supprimer_doublons(leads: List[Dict]) -> List[Dict]:
    """Supprime les doublons basés sur nom + ville"""
    leads_uniques = []
    seen = set()
    
    for lead in leads:
        key = (lead.get('nom', '').lower().strip(), lead.get('ville', '').lower().strip())
        if key not in seen and key[0]:  # Ignorer si nom vide
            seen.add(key)
            leads_uniques.append(lead)
    
    return leads_uniques

logger.info("✓ Fonctions de nettoyage prêtes")

In [ ]:
# 7. EXÉCUTION PRINCIPALE DU SCRAPING

def lancer_scraping() -> pd.DataFrame:
    """Lance le scraping complet avec gestion d'erreur robuste"""
    
    tous_les_leads = []
    
    print("🚀 Démarrage du scraping de leads...")
    print(f"📍 Secteurs: {len(SECTEURS_CIBLES)}")
    print(f"📍 Villes: {len(VILLES_CIBLES)}")
    print(f"📌 Source primaire: API Gouv officielle (France)")
    print(f"📌 Source fallback: OpenCorporates (public)")
    print()
    
    total_iterations = len(SECTEURS_CIBLES) * len(VILLES_CIBLES)
    
    with tqdm(total=total_iterations, desc="Scraping") as pbar:
        
        for secteur in SECTEURS_CIBLES:
            for ville in VILLES_CIBLES:
                try:
                    # API Gouvernementale (PRIMAIRE - très fiable)
                    leads_gouv = scraper_api_gouv(secteur, ville)
                    tous_les_leads.extend(leads_gouv)
                    
                    # OpenCorporates (FALLBACK - pour compléter si peu de résultats)
                    if len(leads_gouv) < 3:
                        leads_oc = scraper_opencorporates_simple(secteur, ville)
                        tous_les_leads.extend(leads_oc)
                
                except Exception as e:
                    logger.error(f"Erreur pour {secteur}/{ville}: {e}")
                
                pbar.update(1)
    
    print()
    print(f"✅ Total leads bruts extraits: {len(tous_les_leads)}")
    
    # Retourner DataFrame vide si aucun lead
    if not tous_les_leads:
        logger.warning("⚠️  Aucun lead extrait! Vérifiez la connexion Internet et les API.")
        print()
        print("🔧 DÉBOGAGE:")
        print("   1. Vérifier: curl https://recherche-entreprises.api.gouv.fr/search?q=restaurant")
        print("   2. Logs détaillés disponibles dans les cellules précédentes")
        print("   3. Vérifier pare-feu/proxy")
        
        return pd.DataFrame(columns=['source', 'nom', 'secteur', 'adresse', 'code_postal', 
                                     'ville', 'telephone', 'email', 'site', 'score', 'date_scrape'])
    
    # Nettoyer les données
    print("🧹 Nettoyage des données...")
    tous_les_leads = [nettoyer_lead(lead) for lead in tous_les_leads]
    
    # Supprimer les doublons
    leads_uniques = supprimer_doublons(tous_les_leads)
    print(f"✅ Après dédoublonnage: {len(leads_uniques)} leads uniques")
    
    # Trier par score décroissant
    leads_uniques.sort(key=lambda x: x.get('score', 0), reverse=True)
    
    # Créer DataFrame
    df = pd.DataFrame(leads_uniques)
    
    # Réorganiser les colonnes
    colonnes = ['source', 'nom', 'secteur', 'adresse', 'code_postal', 'ville', 
                'telephone', 'email', 'site', 'score', 'date_scrape']
    
    df = df[[col for col in colonnes if col in df.columns]]
    
    # Sauvegarder
    df.to_csv(csv_path, index=False, encoding='utf-8-sig')
    
    print()
    print(f"💾 Fichier sauvegardé: {csv_path}")
    print(f"📊 Statistiques:")
    print(f"   - Total leads: {len(df)}")
    
    # Statistiques par source
    if 'source' in df.columns:
        print(f"   - Par source:")
        for source in df['source'].unique():
            count = (df['source'] == source).sum()
            print(f"     • {source}: {count}")
    
    # Gestion des statistiques même si vide
    if len(df) > 0:
        print(f"   - Score moyen: {df['score'].mean():.1f}/100")
        print(f"   - Leads avec téléphone: {(df['telephone'] != '').sum()}")
        print(f"   - Leads avec email: {(df['email'] != '').sum()}")
        print(f"   - Leads avec site: {(df['site'] != '').sum()}")
    
    return df

# Lancer le scraping
df_leads = lancer_scraping()

In [ ]:
# 8. AFFICHAGE DES RÉSULTATS

print()
print("=" * 80)
print("TOP 10 MEILLEURS LEADS")
print("=" * 80)

if len(df_leads) > 0:
    print(df_leads.head(10).to_string())
else:
    print("❌ Aucun lead trouvé - vérifiez votre connexion internet et les sélecteurs CSS")
    print()
    print("💡 Conseils de dépannage:")
    print("   1. Les sites peuvent bloquer les scrapers - attendez quelques minutes")
    print("   2. Essayez en commentant Pages Jaunes et garder seulement OpenCorporates")
    print("   3. Vérifiez votre connexion VPN/Proxy")
    print("   4. Utilisez une IP résidentielle (Pages Jaunes bloque les proxies)")

print()
print("=" * 80)
print(f"Résumé: {len(df_leads)} leads extraits et sauvegardés")
print("=" * 80)

In [ ]:
# 9. VISUALISATIONS ET ANALYSE

import matplotlib.pyplot as plt
import numpy as np

if len(df_leads) > 0:
    fig, axes = plt.subplots(2, 2, figsize=(15, 10))
    fig.suptitle('Analyse des Leads Scrappés', fontsize=16, fontweight='bold')
    
    # 1. Distribution des scores
    ax1 = axes[0, 0]
    ax1.hist(df_leads['score'], bins=20, color='#667eea', edgecolor='black', alpha=0.7)
    ax1.set_xlabel('Score Commercial')
    ax1.set_ylabel('Nombre de leads')
    ax1.set_title('Distribution des Scores')
    ax1.grid(axis='y', alpha=0.3)
    
    # 2. Top secteurs
    ax2 = axes[0, 1]
    top_secteurs = df_leads['secteur'].value_counts().head(5)
    top_secteurs.plot(kind='barh', ax=ax2, color='#764ba2')
    ax2.set_xlabel('Nombre de leads')
    ax2.set_title('Top 5 Secteurs')
    ax2.grid(axis='x', alpha=0.3)
    
    # 3. Données disponibles
    ax3 = axes[1, 0]
    data_available = {
        'Téléphone': (df_leads['telephone'] != '').sum(),
        'Email': (df_leads['email'] != '').sum(),
        'Site web': (df_leads['site'] != '').sum(),
        'Adresse': (df_leads['adresse'] != '').sum()
    }
    ax3.bar(data_available.keys(), data_available.values(), color=['#11998e', '#38ef7d', '#f093fb', '#f5576c'])
    ax3.set_ylabel('Nombre')
    ax3.set_title('Disponibilité des Données de Contact')
    ax3.grid(axis='y', alpha=0.3)
    plt.setp(ax3.xaxis.get_majorticklabels(), rotation=45, ha='right')
    
    # 4. Top villes
    ax4 = axes[1, 1]
    top_villes = df_leads['ville'].value_counts().head(5)
    ax4.pie(top_villes.values, labels=top_villes.index, autopct='%1.1f%%', startangle=90,
            colors=['#667eea', '#764ba2', '#11998e', '#38ef7d', '#f093fb'])
    ax4.set_title('Distribution par Ville')
    
    plt.tight_layout()
    plt.show()
    
    print("\n✅ Visualisations générées avec succès!")
else:
    print("❌ Impossible de générer les visualisations - aucun lead disponible")

In [ ]:
# 10. EXPORT ET FILTRAGE AVANCÉ

if len(df_leads) > 0:
    # Filtrer les leads de haute qualité (score >= 50)
    df_high_quality = df_leads[df_leads['score'] >= 50]
    
    # Filtrer ceux avec téléphone
    df_with_phone = df_leads[df_leads['telephone'] != '']
    
    # Filtrer ceux avec email
    df_with_email = df_leads[df_leads['email'] != '']
    
    print("📊 Filtrage par qualité:")
    print(f"  • Leads haute qualité (score ≥ 50): {len(df_high_quality)}")
    print(f"  • Leads avec téléphone: {len(df_with_phone)}")
    print(f"  • Leads avec email: {len(df_with_email)}")
    
    # Exporter les leads de haute qualité
    high_quality_path = csv_path.replace('.csv', '_high_quality.csv')
    df_high_quality.to_csv(high_quality_path, index=False, encoding='utf-8-sig')
    print(f"\n💾 Leads haute qualité sauvegardés: {high_quality_path}")
    
    # Exporter avec contact
    with_contact_path = csv_path.replace('.csv', '_with_contact.csv')
    df_with_contact = df_leads[(df_leads['telephone'] != '') | (df_leads['email'] != '')]
    df_with_contact.to_csv(with_contact_path, index=False, encoding='utf-8-sig')
    print(f"💾 Leads avec contact sauvegardés: {with_contact_path}")
    
    print(f"\n✅ Tous les fichiers sont prêts à être utilisés dans votre CRM!")

print("\n🎉 Scraping terminé avec succès!")